In [0]:
df = spark.read.csv("/Volumes/workspace/default/chess_data/lichess_2016-06_parsed.csv", header=True, inferSchema=True)
df.printSchema()


root
 |-- white_elo: integer (nullable = true)
 |-- black_elo: integer (nullable = true)
 |-- eco: string (nullable = true)
 |-- opening: string (nullable = true)
 |-- result: string (nullable = true)
 |-- num_moves: integer (nullable = true)
 |-- time_control: string (nullable = true)
 |-- termination: string (nullable = true)
 |-- date: date (nullable = true)



In [0]:
display(df)

white_elo,black_elo,eco,opening,result,num_moves,time_control,termination,date
1601,1793,A02,Bird Opening,1-0,43,120+1,Time forfeit,2016-05-31
1532,1541,C45,Scotch Game,0-1,52,300+0,Normal,2016-05-31
1646,1639,C46,Three Knights Opening,1-0,7,600+0,Normal,2016-05-31
1757,1659,B00,Caro-Kann Defense: Hillbilly Attack,0-1,24,300+0,Normal,2016-05-31
1829,1857,A43,Old Benoni Defense,0-1,46,600+0,Normal,2016-05-31
1508,1501,C41,Philidor Defense #2,1-0,51,600+0,Time forfeit,2016-05-31
1425,1262,A00,Grob Opening,0-1,23,60+2,Time forfeit,2016-05-31
1959,1970,B30,Sicilian Defense: Old Sicilian,0-1,38,600+0,Normal,2016-05-31
2036,2073,D06,Queen's Gambit Refused: Marshall Defense,1-0,51,180+0,Normal,2016-05-31
1578,1661,B00,King's Pawn,1-0,1,600+0,Abandoned,2016-05-31


In [0]:
print("Total rows:", df.count())
print("Missing white_elo:", df.filter(df.white_elo.isNull()).count())
print("Missing black_elo:", df.filter(df.black_elo.isNull()).count())
print("Missing result:", df.filter(df.result.isNull()).count())

Total rows: 6136419
Missing white_elo: 0
Missing black_elo: 0
Missing result: 0


In [0]:
df.groupBy("result").count().show()
df.select("num_moves").summary("min", "25%", "50%", "75%", "max").show()

+-------+-------+
| result|  count|
+-------+-------+
|      *|   1457|
|    1-0|3046465|
|1/2-1/2| 234815|
|    0-1|2853682|
+-------+-------+

+-------+---------+
|summary|num_moves|
+-------+---------+
|    min|        0|
|    25%|       24|
|    50%|       35|
|    75%|       51|
|    max|      493|
+-------+---------+



In [0]:
# how many games are unusually long? worth eyeballing before picking a cutoff
df.filter(df.num_moves > 200).count()

# clean up: drop aborted games and unrealistic outliers
df_clean = df.filter(
    (df.result != "*") &
    (df.num_moves > 0) &
    (df.num_moves <= 200)
)

df_clean.count()

6075566

In [0]:
from pyspark.sql.functions import col, when, abs as spark_abs

df_features = df_clean.withColumn(
    "rating_diff", col("white_elo") - col("black_elo")
).withColumn(
    "higher_rating", when(col("white_elo") >= col("black_elo"), col("white_elo")).otherwise(col("black_elo"))
).withColumn(
    "rating_band", when(col("higher_rating") < 1200, "<1200")
                    .when(col("higher_rating") < 1600, "1200-1600")
                    .when(col("higher_rating") < 2000, "1600-2000")
                    .when(col("higher_rating") < 2400, "2000-2400")
                    .otherwise("2400+")
).withColumn(
    "outcome", when(col("result") == "1-0", "white_win")
                .when(col("result") == "0-1", "black_win")
                .otherwise("draw")
)

df_features.select("white_elo", "black_elo", "rating_diff", "rating_band", "outcome").show(10)

+---------+---------+-----------+-----------+---------+
|white_elo|black_elo|rating_diff|rating_band|  outcome|
+---------+---------+-----------+-----------+---------+
|     1601|     1793|       -192|  1600-2000|white_win|
|     1532|     1541|         -9|  1200-1600|black_win|
|     1646|     1639|          7|  1600-2000|white_win|
|     1757|     1659|         98|  1600-2000|black_win|
|     1829|     1857|        -28|  1600-2000|black_win|
|     1508|     1501|          7|  1200-1600|white_win|
|     1425|     1262|        163|  1200-1600|black_win|
|     1959|     1970|        -11|  1600-2000|black_win|
|     2036|     2073|        -37|  2000-2400|white_win|
|     1578|     1661|        -83|  1600-2000|white_win|
+---------+---------+-----------+-----------+---------+
only showing top 10 rows


### ECO Win-Rate Breakdown

For each opening (grouped by ECO code) and rating band, the White win rate, filtered to openings with at least 500 games in that band to avoid noisy small-sample results.

In [0]:
from pyspark.sql.functions import col, when, count, avg

eco_breakdown = df_features.filter(col("result") != "*") \
    .withColumn("white_won", when(col("outcome") == "white_win", 1).otherwise(0)) \
    .groupBy("eco", "opening", "rating_band") \
    .agg(count("*").alias("games"), avg("white_won").alias("white_win_rate")) \
    .filter(col("games") >= 500) \
    .orderBy(col("games").desc())

eco_breakdown.show(20, truncate=False)



+---+---------------------------------------------+-----------+-----+-------------------+
|eco|opening                                      |rating_band|games|white_win_rate     |
+---+---------------------------------------------+-----------+-----+-------------------+
|A00|Van't Kruijs Opening                         |1600-2000  |70041|0.4313187989891635 |
|B01|Scandinavian Defense: Mieses-Kotroc Variation|1600-2000  |65164|0.5487692591001166 |
|A40|Horwitz Defense                              |1600-2000  |53896|0.5274974024046312 |
|C00|French Defense: Knight Variation             |1600-2000  |51519|0.4691084842485297 |
|B20|Sicilian Defense: Bowdler Attack             |1600-2000  |47454|0.4216293673873646 |
|B00|Owen Defense                                 |1600-2000  |46426|0.5088312583466161 |
|B01|Scandinavian Defense                         |1600-2000  |38528|0.4489721760797342 |
|C41|Philidor Defense #3                          |1600-2000  |37135|0.5662043893900632 |
|A00|Van't

In [0]:
from pyspark.sql.functions import col

# True total games per band (not just the filtered subset), for an accurate percentage
band_totals = df_features.filter(col("result") != "*") \
    .groupBy("rating_band").count().withColumnRenamed("count", "band_total_games")

opening_stats = eco_breakdown.join(band_totals, "rating_band") \
    .withColumn("pct_of_band", col("games") / col("band_total_games") * 100) \
    .select("eco", "opening", "rating_band", "games", "white_win_rate", "pct_of_band") \
    .orderBy("rating_band", col("games").desc())

opening_stats.show(15, truncate=False)

opening_stats.toPandas().to_csv(
    "/Volumes/workspace/default/chess_data/opening_stats.csv", index=False
)

+---+---------------------------------------------+-----------+-----+-------------------+------------------+
|eco|opening                                      |rating_band|games|white_win_rate     |pct_of_band       |
+---+---------------------------------------------+-----------+-----+-------------------+------------------+
|A00|Van't Kruijs Opening                         |1200-1600  |36522|0.4577788730080499 |3.0153243071202174|
|B01|Scandinavian Defense: Mieses-Kotroc Variation|1200-1600  |28829|0.5682125637379029 |2.3801759063021946|
|B01|Scandinavian Defense                         |1200-1600  |27185|0.4604009564097848 |2.2444442059323997|
|C20|King's Pawn Game: Wayward Queen Attack       |1200-1600  |25387|0.5094339622641509 |2.0959979788856296|
|C41|Philidor Defense #3                          |1200-1600  |20124|0.5459650168952495 |1.6614749016069015|
|D00|Queen's Pawn Game #2                         |1200-1600  |19188|0.4890035438815927 |1.5841969992065805|
|C41|Philidor Defen

In [0]:
from pyspark.sql.functions import abs as spark_abs

# Only look at games with a meaningful rating gap — otherwise "favored" barely means anything
df_meaningful = df_features.filter(spark_abs(col("rating_diff")) > 100)

df_meaningful = df_meaningful.withColumn(
    "favored_player_won",
    when(
        ((col("rating_diff") > 0) & (col("outcome") == "white_win")) |
        ((col("rating_diff") < 0) & (col("outcome") == "black_win")),
        1
    ).otherwise(0)
)

df_meaningful.groupBy("rating_band").avg("favored_player_won").orderBy("rating_band").show()

+-----------+-----------------------+
|rating_band|avg(favored_player_won)|
+-----------+-----------------------+
|  1200-1600|     0.6886805283169113|
|  1600-2000|     0.7179608148964555|
|  2000-2400|        0.7476274096573|
|      2400+|     0.7884845066684361|
|      <1200|     0.6576202433841993|
+-----------+-----------------------+



In [0]:
df_meaningful.filter(col("rating_band") == "<1200").count()

10354

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# turn categorical text columns into numeric indices, then one-hot vectors
rating_band_indexer = StringIndexer(inputCol="rating_band", outputCol="rating_band_idx")
opening_indexer = StringIndexer(inputCol="opening", outputCol="opening_idx", handleInvalid="keep")
time_control_indexer = StringIndexer(inputCol="time_control", outputCol="time_control_idx", handleInvalid="keep")
outcome_indexer = StringIndexer(inputCol="outcome", outputCol="label")

encoder = OneHotEncoder(
    inputCols=["rating_band_idx", "opening_idx", "time_control_idx"],
    outputCols=["rating_band_vec", "opening_vec", "time_control_vec"]
)

assembler = VectorAssembler(
    inputCols=["rating_diff", "rating_band_vec", "opening_vec", "time_control_vec"],
    outputCol="features"
)

pipeline = Pipeline(stages=[rating_band_indexer, opening_indexer, time_control_indexer, outcome_indexer, encoder, assembler])
model_data = pipeline.fit(df_features).transform(df_features)

model_data.select("features", "label").show(5, truncate=False)

+-----------------------------------------+-----+
|features                                 |label|
+-----------------------------------------+-----+
|(3766,[0,1,36,2946],[-192.0,1.0,1.0,1.0])|0.0  |
|(3766,[0,3,21,2934],[-9.0,1.0,1.0,1.0])  |1.0  |
|(3766,[0,1,57,2937],[7.0,1.0,1.0,1.0])   |0.0  |
|(3766,[0,1,124,2934],[98.0,1.0,1.0,1.0]) |1.0  |
|(3766,[0,1,30,2937],[-28.0,1.0,1.0,1.0]) |1.0  |
+-----------------------------------------+-----+
only showing top 5 rows


In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
lr_model = lr.fit(train_data)

predictions = lr_model.transform(test_data)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.6339


In [0]:
from pyspark.sql.functions import lit

# naive baseline: always predict the favored player wins
baseline = test_data.withColumn(
    "baseline_correct",
    when(
        ((col("rating_diff") > 0) & (col("outcome") == "white_win")) |
        ((col("rating_diff") < 0) & (col("outcome") == "black_win")),
        1
    ).otherwise(0)
)

baseline_accuracy = baseline.filter(col("rating_diff") != 0).agg({"baseline_correct": "avg"}).collect()[0][0]
print(f"Naive baseline accuracy: {baseline_accuracy:.4f}")
print(f"Model accuracy: {accuracy:.4f}")
print(f"Improvement: {(accuracy - baseline_accuracy) * 100:.2f} percentage points")

Naive baseline accuracy: 0.6320
Model accuracy: 0.6339
Improvement: 0.19 percentage points


In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)

rf_accuracy = evaluator.evaluate(rf_predictions)
print(f"Random Forest accuracy: {rf_accuracy:.4f}")
print(f"Logistic Regression accuracy: {accuracy:.4f}")
print(f"Naive baseline accuracy: {baseline_accuracy:.4f}")

Random Forest accuracy: 0.6301
Logistic Regression accuracy: 0.6339
Naive baseline accuracy: 0.6320


In [0]:
from pyspark.sql.functions import regexp_extract, col, expr

df_cluster = df_features.withColumn(
    "base_time_str", regexp_extract(col("time_control"), r"^(\d+)\+", 1)
).withColumn(
    "increment_str", regexp_extract(col("time_control"), r"\+(\d+)$", 1)
).withColumn(
    "base_time", expr("try_cast(base_time_str as int)")
).withColumn(
    "increment", expr("try_cast(increment_str as int)")
)

df_cluster = df_cluster.filter(col("base_time").isNotNull() & col("increment").isNotNull())
df_cluster.count()

6052543

In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

termination_indexer = StringIndexer(inputCol="termination", outputCol="termination_idx", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["higher_rating", "num_moves", "base_time", "increment", "termination_idx"],
    outputCol="raw_features"
)

scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)

kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=4, seed=42)

pipeline = Pipeline(stages=[termination_indexer, assembler, scaler, kmeans])
cluster_model = pipeline.fit(df_cluster)
clustered = cluster_model.transform(df_cluster)

clustered.groupBy("cluster").count().orderBy("cluster").show()

+-------+-------+
|cluster|  count|
+-------+-------+
|      0|1749961|
|      1|3575294|
|      2| 588980|
|      3| 138308|
+-------+-------+



In [0]:
clustered.groupBy("cluster").agg(
    {"higher_rating": "avg", "num_moves": "avg", "base_time": "avg", "increment": "avg"}
).orderBy("cluster").show()

clustered.groupBy("cluster", "termination").count().orderBy("cluster", "count", ascending=[True, False]).show(20)

+-------+------------------+------------------+------------------+------------------+
|cluster|avg(higher_rating)|    avg(num_moves)|    avg(base_time)|    avg(increment)|
+-------+------------------+------------------+------------------+------------------+
|      0|1832.5058947027962|38.594128097711895| 196.5483773638384|0.5507842746209772|
|      1|1809.2106724649777|33.075838798151985|    317.2227542686|1.1692747505519825|
|      2| 1831.406312608238|123.54508132703997| 277.9930048558525|1.2860385751638426|
|      3|1674.6752827023745| 40.91662810538797|1675.7781184024063|11.634995806460942|
+-------+------------------+------------------+------------------+------------------+

+-------+----------------+-------+
|cluster|     termination|  count|
+-------+----------------+-------+
|      0|    Time forfeit|1738352|
|      0|       Abandoned|  11461|
|      0|Rules infraction|    148|
|      1|          Normal|3575294|
|      2|          Normal| 421786|
|      2|    Time forfeit| 1671

In [0]:
from pyspark.sql.functions import col, split, posexplode, when, regexp_extract, abs as spark_abs

# 1. Load the evaluated-games CSV
df_eval = spark.read.csv(
    "/Volumes/workspace/default/chess_data/lichess_2016-06_evaluated_games.csv",
    header=True, inferSchema=True
)

# 2. Split the evals string into an array, then explode into one row per move
df_exploded = df_eval.withColumn("eval_array", split(col("evals"), ";")) \
                      .select("*", posexplode(col("eval_array")).alias("move_number", "eval_str"))

# 3. Parse each eval value: plain numbers vs mate markers like "#-4"
df_parsed = df_exploded.withColumn(
    "is_mate", col("eval_str").startswith("#")
).withColumn(
    "eval_pawns",
    when(col("is_mate") & col("eval_str").startswith("#-"), -100.0)   # losing forced mate
    .when(col("is_mate"), 100.0)                                       # winning forced mate
    .otherwise(col("eval_str").cast("double"))
).withColumn(
    "eval_centipawns", col("eval_pawns") * 100   # match the same units the demo script uses
)

# 4. Attach rating band + outcome, same logic as your main dataset
df_labeled = df_parsed.withColumn(
    "higher_rating", when(col("white_elo") >= col("black_elo"), col("white_elo")).otherwise(col("black_elo"))
).withColumn(
    "rating_band", when(col("higher_rating") < 1200, "<1200")
                    .when(col("higher_rating") < 1600, "1200-1600")
                    .when(col("higher_rating") < 2000, "1600-2000")
                    .when(col("higher_rating") < 2400, "2000-2400")
                    .otherwise("2400+")
).withColumn(
    "outcome", when(col("result") == "1-0", "white_win")
                .when(col("result") == "0-1", "black_win")
                .otherwise("draw")
).filter(col("result") != "*")

# 5. Bucket the evaluation into ranges, from White's perspective
df_bucketed = df_labeled.withColumn(
    "eval_bucket",
    when(col("eval_centipawns") < -500, "< -500")
    .when(col("eval_centipawns") < -300, "-500 to -300")
    .when(col("eval_centipawns") < -100, "-300 to -100")
    .when(col("eval_centipawns") < 100, "-100 to +100")
    .when(col("eval_centipawns") < 300, "+100 to +300")
    .when(col("eval_centipawns") < 500, "+300 to +500")
    .otherwise("> +500")
).withColumn(
    "white_won", when(col("outcome") == "white_win", 1).otherwise(0)
)

# 6. The actual calibration table: real win% by eval bucket AND rating band
calibration_table = df_bucketed.groupBy("eval_bucket", "rating_band") \
    .agg({"white_won": "avg"}) \
    .orderBy("eval_bucket", "rating_band")

calibration_table.show(40)

+------------+-----------+-------------------+
| eval_bucket|rating_band|     avg(white_won)|
+------------+-----------+-------------------+
|+100 to +300|  1200-1600| 0.6023010420220323|
|+100 to +300|  1600-2000| 0.6314201834159113|
|+100 to +300|  2000-2400| 0.6762301944450259|
|+100 to +300|      2400+| 0.7013911547474269|
|+100 to +300|      <1200| 0.5646377968469367|
|+300 to +500|  1200-1600| 0.6726116091931925|
|+300 to +500|  1600-2000| 0.7132278144260206|
|+300 to +500|  2000-2400| 0.7751949926561208|
|+300 to +500|      2400+| 0.8126401630988787|
|+300 to +500|      <1200|  0.608605648380896|
|-100 to +100|  1200-1600|  0.482632863473985|
|-100 to +100|  1600-2000|0.47672853530987547|
|-100 to +100|  2000-2400| 0.4686072391580507|
|-100 to +100|      2400+| 0.4532468827561754|
|-100 to +100|      <1200|0.47582709095371095|
|-300 to -100|  1200-1600| 0.3633348213018606|
|-300 to -100|  1600-2000| 0.3243015607257699|
|-300 to -100|  2000-2400|0.26683760995359823|
|-300 to -100

In [0]:
calibration_table.toPandas().to_csv("/Workspace/Users/fbite0897@gmail.com/chess_data_engineeringcalibration_table.csv", index=False)

In [0]:
df_features.select("higher_rating").summary("min", "25%", "50%", "75%", "max").show()
df_features.select("num_moves").summary("min", "25%", "50%", "75%", "max").show()

rating_hist = df_features.groupBy("rating_band").count().orderBy("rating_band")
rating_hist.show()

+-------+-------------+
|summary|higher_rating|
+-------+-------------+
|    min|          823|
|    25%|         1640|
|    50%|         1811|
|    75%|         1983|
|    max|         3069|
+-------+-------------+

+-------+---------+
|summary|num_moves|
+-------+---------+
|    min|        1|
|    25%|       24|
|    50%|       35|
|    75%|       51|
|    max|      200|
+-------+---------+

+-----------+-------+
|rating_band|  count|
+-----------+-------+
|  1200-1600|1211213|
|  1600-2000|3435543|
|  2000-2400|1314822|
|      2400+|  80214|
|      <1200|  33774|
+-----------+-------+



### Turning-Point Detection

For each evaluated game, finds the single move where the position's evaluation swung the hardest in one step — the moment the game most decisively turned. Uses a game_id (assigned here since the evaluated-games CSV doesn't have one) to group each game's moves together, then a window function to compare each move's evaluation to the one immediately before it.

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col, split, posexplode, when, lag, row_number
from pyspark.sql.window import Window

# Reload with a unique ID per game
df_eval_id = spark.read.csv(
    "/Volumes/workspace/default/chess_data/lichess_2016-06_evaluated_games.csv",
    header=True, inferSchema=True
).withColumn("game_id", monotonically_increasing_id())

df_exploded_tp = df_eval_id.withColumn("eval_array", split(col("evals"), ";")) \
    .select("game_id", "white_elo", "black_elo", "result", "opening",
            posexplode(col("eval_array")).alias("move_number", "eval_str"))

df_parsed_tp = df_exploded_tp.withColumn(
    "eval_pawns",
    when(col("eval_str").startswith("#-"), -100.0)
    .when(col("eval_str").startswith("#"), 100.0)
    .otherwise(col("eval_str").cast("double"))
).withColumn("eval_centipawns", col("eval_pawns") * 100)

# Compare each move's eval to the one before it, within the same game
w = Window.partitionBy("game_id").orderBy("move_number")
df_swings = df_parsed_tp.withColumn("prev_eval", lag("eval_centipawns").over(w)) \
    .withColumn("swing", col("eval_centipawns") - col("prev_eval")) \
    .filter(col("prev_eval").isNotNull()) \
    .withColumn("abs_swing", when(col("swing") < 0, -col("swing")).otherwise(col("swing")))

# Keep only the single biggest swing per game — that's the turning point
w2 = Window.partitionBy("game_id").orderBy(col("abs_swing").desc())
turning_points = df_swings.withColumn("rn", row_number().over(w2)).filter(col("rn") == 1).drop("rn")

turning_points.select("game_id", "white_elo", "black_elo", "opening", "move_number", "swing", "result") \
    .orderBy(col("abs_swing").desc()).show(10)

+-----------+---------+---------+--------------------+-----------+--------+-------+
|    game_id|white_elo|black_elo|             opening|move_number|   swing| result|
+-----------+---------+---------+--------------------+-----------+--------+-------+
| 8589992275|     1731|     1256|Sicilian Defense:...|        115| 23423.0|    1-0|
|      67609|     1642|     1414|Englund Gambit De...|        110|-22352.0|    0-1|
|      11069|     1809|     1339|Rat Defense: Smal...|         91| 22349.0|    1-0|
|     178627|     1944|     1970|Caro-Kann Defense...|         77| 22347.0|    1-0|
|17180032093|     2149|     1958|Semi-Slav Defense...|         85| 22343.0|    1-0|
| 8589987822|     1642|     1541|Three Knights Ope...|        134|-22330.0|    0-1|
|25769888756|     1583|     1635|Italian Game: Giu...|         86|-22328.0|1/2-1/2|
|17179930976|     2419|     2342|Sicilian Defense:...|        111| 22320.0|    1-0|
| 8590114961|     2133|     1908|Zukertort Opening...|         78|-22183.0| 

### Opening Win-Rate Spread Across Rating Bands

Pivots the ECO breakdown so each opening's win rate is visible across all five rating bands in one row, to find openings that behave very differently depending on skill level.

In [0]:
from pyspark.sql.functions import avg

# This is the missing step - eco_breakdown pivoted so each opening has one
# win-rate column per rating band, side by side, which pivoted_clean below needs.
pivoted = eco_breakdown.groupBy("eco", "opening").pivot("rating_band").agg(avg("white_win_rate"))
pivoted.show(5, truncate=False)

+---+---------------------------------------------+-------------------+------------------+-------------------+-------------------+------------------+
|eco|opening                                      |1200-1600          |1600-2000         |2000-2400          |2400+              |<1200             |
+---+---------------------------------------------+-------------------+------------------+-------------------+-------------------+------------------+
|A00|Van't Kruijs Opening                         |0.4577788730080499 |0.4313187989891635|0.43656864693026026|0.45602365114560234|0.4890337877889745|
|B01|Scandinavian Defense: Mieses-Kotroc Variation|0.5682125637379029 |0.5487692591001166|0.5539483701294208 |0.5301204819277109 |NULL              |
|A40|Horwitz Defense                              |0.5068664169787765 |0.5274974024046312|0.5296153846153846 |0.5664251207729468 |NULL              |
|C00|French Defense: Knight Variation             |0.4934934934934935 |0.4691084842485297|0.40634866

In [0]:
from pyspark.sql.functions import col, abs as spark_abs

pivoted_clean = pivoted.filter(col("1200-1600").isNotNull() & col("2000-2400").isNotNull())
pivoted_clean = pivoted_clean.withColumn("spread", col("2000-2400") - col("1200-1600"))
pivoted_clean.withColumn("abs_spread", spark_abs(col("spread"))) \
    .orderBy(col("abs_spread").desc()) \
    .select("eco", "opening", "1200-1600", "2000-2400", "spread") \
    .show(10, truncate=False)

+---+---------------------------------------------------------------+-------------------+-------------------+--------------------+
|eco|opening                                                        |1200-1600          |2000-2400          |spread              |
+---+---------------------------------------------------------------+-------------------+-------------------+--------------------+
|C40|King's Pawn Game: McConnell Defense                            |0.527619769940589  |0.7410161090458488 |0.2133963391052598  |
|A06|Zukertort Opening: Reversed Mexican Defense                    |0.38629876308277833|0.5887323943661972 |0.20243363128341885 |
|C21|Center Game #2                                                 |0.5580530674328207 |0.7416974169741697 |0.183644349541349   |
|C40|King's Pawn Game: Busch-Gass Gambit                            |0.5418840579710145 |0.7223880597014926 |0.1805040017304781  |
|C22|Center Game: Paulsen Attack Variation                          |0.416422287390

In [0]:
df_swings_clean = df_swings.filter(
    (spark_abs(col("eval_centipawns")) < 5000) & (spark_abs(col("prev_eval")) < 5000)
)
w3 = Window.partitionBy("game_id").orderBy(col("abs_swing").desc())
turning_points_clean = df_swings_clean.withColumn("rn", row_number().over(w3)).filter(col("rn") == 1).drop("rn")
turning_points_clean.select("game_id", "white_elo", "black_elo", "opening", "move_number", "swing", "result") \
    .orderBy(col("abs_swing").desc()).show(10)

+-----------+---------+---------+--------------------+-----------+-------+-------+
|    game_id|white_elo|black_elo|             opening|move_number|  swing| result|
+-----------+---------+---------+--------------------+-----------+-------+-------+
|34359767740|     1752|     1824|Nimzo-Indian Defe...|        102|-9910.0|    0-1|
| 8589937689|     2135|     2063|   Caro-Kann Defense|        118|-9906.0|    1-0|
| 8590016109|     1355|     1630|    Sicilian Defense|         86|-9898.0|    0-1|
|42949746920|     1894|     1590|Scandinavian Defe...|         83| 9893.0|    1-0|
|34359787309|     2347|     2352|Sicilian Defense:...|        117| 9875.0|    1-0|
|17179943836|     1873|     1621|Caro-Kann Defense...|        103| 9867.0|1/2-1/2|
|      67495|     1522|     1528|King's Pawn Game:...|         79| 9840.0|    1-0|
|34359784020|     1591|     1640|Sicilian Defense:...|        116|-9837.0|    0-1|
|      82104|     1919|     1705|Scandinavian Defe...|        111| 9836.0|    1-0|
|429

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/chess_data/")

[FileInfo(path='dbfs:/Volumes/workspace/default/chess_data/lichess_2016-06_evaluated_games.csv', name='lichess_2016-06_evaluated_games.csv', size=297815603, modificationTime=1787469661000),
 FileInfo(path='dbfs:/Volumes/workspace/default/chess_data/lichess_2016-06_parsed.csv', name='lichess_2016-06_parsed.csv', size=490882196, modificationTime=1787116134000),
 FileInfo(path='dbfs:/Volumes/workspace/default/chess_data/opening_stats.csv', name='opening_stats.csv', size=163429, modificationTime=1789048430000)]

In [0]:
example_game_id = turning_points.orderBy(col("abs_swing").desc()).first()["game_id"]

trajectory = df_swings.filter(col("game_id") == example_game_id) \
    .select("move_number", "eval_centipawns").orderBy("move_number")

trajectory.show(50)
# Convert trajectory.toPandas() to a line chart in your deck/report

+-----------+------------------+
|move_number|   eval_centipawns|
+-----------+------------------+
|          1|              34.0|
|          2|               8.0|
|          3|              10.0|
|          4|              10.0|
|          5|              10.0|
|          6|             -10.0|
|          7|             -10.0|
|          8|              -8.0|
|          9|              15.0|
|         10|              10.0|
|         11|14.000000000000002|
|         12|              12.0|
|         13|              12.0|
|         14|              18.0|
|         15|              24.0|
|         16|              18.0|
|         17|             148.0|
|         18|             169.0|
|         19|             187.0|
|         20|             162.0|
|         21|224.00000000000003|
|         22|             166.0|
|         23|             359.0|
|         24|245.00000000000003|
|         25|             560.0|
|         26|              66.0|
|         27|              64.0|
|         

In [0]:
from pyspark.sql.functions import when, col

df_labeled_swings = df_swings.withColumn("higher_rating", when(col("white_elo") >= col("black_elo"), col("white_elo")).otherwise(col("black_elo"))) \
    .withColumn("rating_band", when(col("higher_rating") < 1200, "<1200")
                .when(col("higher_rating") < 1600, "1200-1600")
                .when(col("higher_rating") < 2000, "1600-2000")
                .when(col("higher_rating") < 2400, "2000-2400")
                .otherwise("2400+")) \
    .withColumn("severity", when(col("abs_swing") >= 300, "blunder")
                .when(col("abs_swing") >= 150, "mistake")
                .when(col("abs_swing") >= 50, "inaccuracy")
                .otherwise("ok"))

df_labeled_swings.filter(col("severity") != "ok") \
    .groupBy("rating_band", "severity").count() \
    .orderBy("rating_band", "severity").show(20)
    

+-----------+----------+-------+
|rating_band|  severity|  count|
+-----------+----------+-------+
|  1200-1600|   blunder|1053120|
|  1200-1600|inaccuracy|1507475|
|  1200-1600|   mistake| 600768|
|  1600-2000|   blunder|2299908|
|  1600-2000|inaccuracy|4064372|
|  1600-2000|   mistake|1452153|
|  2000-2400|   blunder| 769588|
|  2000-2400|inaccuracy|1534531|
|  2000-2400|   mistake| 510859|
|      2400+|   blunder|  57380|
|      2400+|inaccuracy| 119976|
|      2400+|   mistake|  38154|
|      <1200|   blunder|  44973|
|      <1200|inaccuracy|  46344|
|      <1200|   mistake|  21672|
+-----------+----------+-------+

